# 05 - Tableau Data Preparation

**AI Forensic Triage Tool: Predicting Shooting Incidents in Boston**  
**Capstone Project — DSE 6311**  
**Author**: Ricardo Orellana  

**Goal**: Load the final tuned model and processed data, generate predictions + SHAP values, and export a clean CSV ready for the interactive Tableau dashboard.

In [9]:
import pandas as pd
import xgboost as xgb
import shap
import requests
from io import BytesIO
from pathlib import Path
import tempfile
import os

print("✅ Starting Tableau Data Preparation (Notebook 05)")

✅ Starting Tableau Data Preparation (Notebook 05)


## 1. Load Data & Model from GitHub (Team-Friendly)

In [11]:
# ================== RAW GITHUB URLS ==================
BASE = "https://raw.githubusercontent.com/Rick-997/AI-Forensics-Boston-Capstone/main"

DATA_URL = f"{BASE}/data/processed/crimes_cleaned.parquet"
MODEL_URL = f"{BASE}/models/xgboost_shooting_model.json"

# ================== LOAD DATA ==================
r_data = requests.get(DATA_URL)
r_data.raise_for_status()
df = pd.read_parquet(BytesIO(r_data.content))
print(f"✅ Data loaded — {len(df):,} rows")

# ================== ONE-HOT ENCODE DISTRICT ==================
district_dummies = pd.get_dummies(df['DISTRICT'], prefix='district')

expected_districts = [
    'district_A15', 'district_A7', 'district_B2', 'district_B3',
    'district_C11', 'district_C6', 'district_D14', 'district_D4',
    'district_E13', 'district_E18', 'district_E5',
    'district_External', 'district_Outside of', 'district_Unknown'
]

for col in expected_districts:
    if col not in district_dummies.columns:
        district_dummies[col] = 0

df = pd.concat([df, district_dummies[expected_districts]], axis=1)

df['poverty_rate'] = 0.20   # model expects this column
print("✅ DISTRICT one-hot encoded + poverty_rate added")

✅ Data loaded — 239,371 rows
✅ DISTRICT one-hot encoded + poverty_rate added


## 2. Load Final Tuned Model

In [13]:
r_model = requests.get(MODEL_URL)
r_model.raise_for_status()

with tempfile.NamedTemporaryFile(suffix='.json', delete=False) as tmp:
    tmp.write(r_model.content)
    tmp_path = tmp.name

model = xgb.Booster()
model.load_model(tmp_path)
os.unlink(tmp_path)

print("✅ Final tuned model loaded successfully from GitHub")

✅ Final tuned model loaded successfully from GitHub


## 3. Generate Predictions + SHAP Values

In [15]:
feature_cols = ['hour', 'is_night', 'is_weekend', 'is_violent', 'poverty_rate'] + expected_districts

X = df[feature_cols]
df['predicted_prob'] = model.predict(xgb.DMatrix(X))

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

for i, col in enumerate(feature_cols[:8]):
    df[f'shap_{col}'] = shap_values[:, i]

print("✅ Predictions + top SHAP values added")

✅ Predictions + top SHAP values added


## 4. Save Tableau-Ready CSV

In [17]:
TABLEAU_FOLDER = Path("../visualizations/tableau")
TABLEAU_FOLDER.mkdir(parents=True, exist_ok=True)

output_file = TABLEAU_FOLDER / "tableau_ready.csv"
df.to_csv(output_file, index=False)

print("\n🎉 SUCCESS! Tableau-ready file created")
print(f"   Location: {output_file}")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {len(df.columns)}")
print("\nYou can now open this CSV directly in Tableau!")


🎉 SUCCESS! Tableau-ready file created
   Location: ..\visualizations\tableau\tableau_ready.csv
   Rows    : 239,371
   Columns : 38

You can now open this CSV directly in Tableau!
